# FedCrohn — Severity-Aware Extension

Multi-task GAT predicting Crohn's **diagnosis** and **SES-CD severity**,
federated across 4 real hospitals (HMP2 host transcriptomics).

## How to run

1. Edit **CONFIG** (next cell) for the experiment you want.
2. Run **Setup** cells in order.
3. Run **one** experiment cell.

**Every config change needs a kernel restart.** Cell 6 wraps `__init__`;
running it twice in one session double-wraps it. Restart → edit CONFIG →
rerun setup.

| experiment | CONFIG needed |
|---|---|
| 1. Main result (3 seeds) | defaults |
| 2. Centralized baseline | defaults |
| 3. Feature ablation | defaults |
| 4. Per-site breakdown | defaults (included in exp 1) |
| 5. Non-IID table | none — no training |
| 6. Severity XAI | defaults |
| 7. 3-level severity | `N_SEV_LEVELS = 3` |
| 8. Gene subsetting | defaults |
| 9. Differential privacy | `DP_SIGMA = 1.5` |
| 10. Pooling ablation | `POOL_GENES = True` |

Results from previous runs: [`results/RESULTS_SUMMARY.md`](results/RESULTS_SUMMARY.md)

## CONFIG — the only cell you edit

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────
# Change these, then RESTART THE KERNEL and rerun the setup cells.

PROJECT_PATH  = "/kaggle/input/datasets/bhanavi1231/hmp2-severity"

N_SEV_LEVELS  = 4       # 4 = clinical SES-CD cutoffs; 3 = merge moderate+severe
DP_SIGMA      = 0.0     # 0 = no DP. >0 enables per-example DP-SGD
POOL_GENES    = False   # True = shrink the head (ablation only; destroys utility)
POOL_MODE     = "mean"  # "mean" or "attention", only used when POOL_GENES=True

# Defaults for the experiment cells below
LAM           = 0.5     # severity loss weight
REGION_ZSCORE = True    # z-score genes within biopsy region (helps severity)
REGION_FLAG   = False   # append ileum indicator     (helps diagnosis)

print(f"n_levels={N_SEV_LEVELS}  dp_sigma={DP_SIGMA}  "
      f"pool={POOL_GENES}({POOL_MODE})  lam={LAM}")

## Setup — run cells 1–7 in order

In [ ]:
# Cell 1 — Setup
import sys, os
sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

import torch, numpy as np, scipy
print("CWD:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("numpy:", np.__version__, "| scipy:", scipy.__version__)

In [ ]:
# Cell 2 — Verify files & imports
import os, pickle

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.loads(f.read().replace(b"\r\n", b"\n"), encoding="latin1")

for f in ["marshalledP3/totGeneSet.m.min0", "marshalledP3/adj_cache.pkl",
          "phenopediaCrohnGenes/CrohnGenes.txt", "host_tx_counts.tsv",
          "hmp2_metadata_2018-08-20.csv", "hmp2_loader.py",
          "severity_model.py", "run_severity_fl.py", "severity_xai.py"]:
    ok = os.path.exists(f)
    print(f"{'OK' if ok else 'MISSING':8s} "
          f"{f'{os.path.getsize(f)/1e6:.1f} MB' if ok else '':10s} {f}")

from sources.GATmodel import GATCrohnModel
from sources.buildGeneGraph import build_adj_from_string, build_adj_phenopedia
import sources.GraphConv as GCN
print("\nimports OK")

In [ ]:
# Cell 3 — Gene graph (loads from cache; string_db/ not needed)
from sources.readPhenopedia import readPhenopedia
import numpy as np

geneList = sorted(load_pkl("marshalledP3/totGeneSet.m.min0"))
weightPhenoPGenes, _ = readPhenopedia("phenopediaCrohnGenes/CrohnGenes.txt")

adj = build_adj_from_string(
    geneList,
    string_links_path="string_db/9606.protein.links.v12.0.txt",
    string_info_path="string_db/9606.protein.info.v12.0.txt",
    threshold=700, cache_path="marshalledP3/adj_cache.pkl")

if adj is None or np.array_equal(adj, np.eye(len(geneList))):
    adj = build_adj_phenopedia(geneList, weightPhenoPGenes)

print(f"genes {len(geneList)} | adj {adj.shape} | density {(adj>0).mean():.4f}")

In [ ]:
# Cell 4 — Globals & seeds
import numpy as np, torch as t, random as _r
from sources import utils as U

DEVICE = t.device("cuda" if t.cuda.is_available() else "cpu")
_r.seed(42); np.random.seed(42); t.manual_seed(42); t.cuda.manual_seed_all(42)
os.makedirs("/kaggle/working/results", exist_ok=True)
print("device:", DEVICE)

In [ ]:
# Cell 5 — Memory-efficient GAT attention
# Peak alloc [B,N,N] per head instead of [B,N,N,H]. At B=4, N=691 that is
# ~7 MB per head vs ~3.6 GB.
import torch, torch.nn.functional as F
from sources.GATmodel import GATLayer, GATCrohnModel

def _efficient_gat_forward(self, x, adj):
    B, N, _ = x.shape
    h = self.W(x).view(B, N, self.num_heads, self.out_features)
    outs, last = [], None
    for k in range(self.num_heads):
        hk = h[:, :, k, :]
        e = self.leakyrelu(torch.cat([
            hk.unsqueeze(2).expand(B, N, N, self.out_features),
            hk.unsqueeze(1).expand(B, N, N, self.out_features)], -1
        ).matmul(self.a[k]))
        alpha = F.softmax(e.masked_fill((adj == 0).unsqueeze(0), float("-inf")), 2)
        alpha = self.dropout(alpha)
        outs.append(torch.bmm(alpha, hk)); last = alpha.detach()
        del e, alpha; torch.cuda.empty_cache()
    return torch.stack(outs, 0).mean(0), last

def _gene_importance(self):
    return None if self.attention_weights is None else \
        self.attention_weights.mean(dim=0).sum(dim=0)

GATLayer.forward = _efficient_gat_forward
GATCrohnModel.get_gene_importance = _gene_importance
print("GATLayer patched")

In [ ]:
# Cell 6 — Severity heads + DP   ***RUN ONCE PER SESSION***
# Wraps __init__. Running twice double-wraps it — restart the kernel instead.
from severity_model import (attach_severity_heads, patch_wrapper_multitask,
                            fedavg_multitask, severity_metrics,
                            diagnosis_metrics, compute_epsilon)
from sources.GraphConv import NNwrapper

if hasattr(GATCrohnModel, "_severity_attached"):
    del GATCrohnModel._severity_attached

attach_severity_heads(GATCrohnModel, n_sev_levels=N_SEV_LEVELS,
                      pool_genes=POOL_GENES, pool_mode=POOL_MODE)
GATCrohnModel._severity_attached = True

patch_wrapper_multitask(NNwrapper, DEVICE, dp_sigma=DP_SIGMA,
                        dp_per_example=True)

print(f"heads attached: CORAL {N_SEV_LEVELS} levels, "
      f"{'POOLED-' + POOL_MODE if POOL_GENES else 'FLATTEN'}")
print(f"NNwrapper patched: dp_sigma={DP_SIGMA}")
print("verify:", GATCrohnModel(3, 691, adj, geneList).n_sev_levels, "levels")

In [ ]:
# Cell 7 — Load HMP2
from hmp2_loader import build_hmp2_dataset, make_node_features

data = build_hmp2_dataset(
    meta_path   = "hmp2_metadata_2018-08-20.csv",
    counts_path = "host_tx_counts.tsv",
    geneList    = geneList,
    n_levels    = N_SEV_LEVELS,
)

---
## Experiment 1 — Main result, 3 seeds  *(~3 h)*

**CONFIG:** defaults.

A single seed is not reproducible — QWK ranged 0.18–0.31 across seeds — so all
reported numbers are seed-averaged. Also emits the per-site breakdown.

In [ ]:
from run_severity_fl import run_federated_severity, save_results
import numpy as np

runs = {}
for sd in [42, 7, 123]:
    print(f"\n{'#'*60}\nSEED {sd}\n{'#'*60}")
    res, sites = run_federated_severity(
        data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
        n_folds=5, num_rounds=5, epochs_per_client=50, lam=LAM,
        region_zscore=REGION_ZSCORE, region_flag=REGION_FLAG, seed=sd)
    runs[sd] = (res, sites)
    save_results((res, sites), tag=f"regionz_seed{sd}")

print("\n=== ACROSS SEEDS ===")
for k in ["mcc", "auc", "sev_spearman", "sev_qwk", "sev_mae_levels"]:
    v = [np.mean([f[k] for f in r[0] if k in f and f[k] == f[k]])
         for r in runs.values()]
    print(f"  {k:18s} {np.mean(v):.4f} +/- {np.std(v):.4f}   {np.round(v,3)}")

---
## Experiment 2 — Centralized baseline  *(~30 min)*

**CONFIG:** defaults. The privacy–utility comparison.

Matched to Experiment 1 on folds, seed, features, `lam`, and the 5-checkpoint
selection protocol. Without matching the checkpointing, federated gets
best-of-5 on the test set and centralized gets one shot — which inverts the
result.

In [ ]:
from run_severity_fl import run_centralized_baseline

res_central = run_centralized_baseline(
    data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
    n_folds=5, epochs=100, lam=LAM,
    region_zscore=REGION_ZSCORE, region_flag=REGION_FLAG)

---
## Experiment 3 — Feature ablation  *(~3 h, 1 h each)*

**CONFIG:** defaults.

Biopsy region (ileum vs colorectal, ~50/50 in HMP2) is independent of both
diagnosis and severity (chi2 p~0.47), so it is not a confound — but it is
variance. Tissue identity *helps* diagnosis; removing tissue variation *helps*
severity. Opposite operations, so no single config wins both.

In [ ]:
KW = dict(n_folds=5, num_rounds=5, epochs_per_client=50, lam=LAM, seed=42)

for tag, kw in [("baseline",   dict()),
                ("regionz",    dict(region_zscore=True)),
                ("regionflag", dict(region_flag=True)),
                ("regionboth", dict(region_zscore=True, region_flag=True))]:
    print(f"\n{'#'*60}\n{tag}\n{'#'*60}")
    r, s = run_federated_severity(data, GATCrohnModel, NNwrapper, adj,
                                  geneList, DEVICE, **KW, **kw)
    save_results((r, s), tag=tag)

---
## Experiment 4 — Non-IID site distribution  *(instant)*

**CONFIG:** none — no training. Run after Cell 7.

Cincinnati is 51% moderate-severity; Cedars-Sinai is 0%. Real measured
heterogeneity across four hospitals, and it explains the per-site results:
three of four clients are remission-dominated, so FedAvg follows them.

In [ ]:
import pandas as pd

lab = data["labels"]
cd  = lab[lab.sev_mask == 1]
tab = pd.crosstab(cd["site"], cd["y_sev"])
tab.columns = ["remission 0-2", "mild 3-6", "moderate 7-15",
               "severe 16+"][:tab.shape[1]]
print(tab.to_string())
print("\nrow % (severity mix per site):")
print((tab.div(tab.sum(axis=1), axis=0) * 100).round(1).to_string())
tab.to_csv("/kaggle/working/results/site_severity_distribution.csv")

---
## Experiment 5 — Severity-specific gene attribution  *(~20 min)*

**CONFIG:** defaults.

GAT attention sits upstream of the head split, so it cannot separate the two
tasks. This does gradient attribution on each head separately and contrasts
them: a gene that moves severity but not diagnosis is severity-specific.

`mean_pairwise_overlap` decides whether the list is reportable. Above ~0.5,
name only the genes present in the top-50 for every seed.

In [ ]:
from severity_xai import (severity_gene_importance, save_gene_report,
                          stability_across_seeds)
from run_severity_fl import participant_folds
import torch as t

dfs = []
for sd in [42, 7, 123]:
    tr, te = participant_folds(data["labels"], 5, sd)[0]
    X = make_node_features(data["raw_counts"], train_idx=tr,
                           region=data["region"],
                           region_zscore=REGION_ZSCORE,
                           region_flag=REGION_FLAG)
    net = GATCrohnModel(X.shape[2], X.shape[1], adj, geneList).to(DEVICE)
    w = NNwrapper(net)
    w.fit([X[i] for i in tr], [data["Y"][i] for i in tr],
          epochs=50, batch_size=4, lam=LAM, silent=True)
    dfs.append(severity_gene_importance(net, X, data["Y"], geneList, DEVICE))
    del net, w; t.cuda.empty_cache()

save_gene_report(dfs[0])
st = stability_across_seeds(dfs, top_n=50)
print("\n=== STABILITY ===")
print({k: v for k, v in st.items() if k != "in_all_seeds"})
print("stable genes:", st["in_all_seeds"])

---
## Experiment 6 — Three-level severity  *(~1 h)*

**CONFIG:** `N_SEV_LEVELS = 3`, then restart and rerun setup.

Merges moderate+severe into "active disease" (58/26/34 instead of
58/26/25/10). Improves per-class recall on the merged class (41% vs 8%) but
lowers overall rank correlation. Report as a robustness analysis, not the
primary result — and note QWK/MAE are not comparable across binnings; Spearman
is.

In [ ]:
assert N_SEV_LEVELS == 3, "set N_SEV_LEVELS = 3 in CONFIG, restart, rerun setup"

res3, s3 = run_federated_severity(
    data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
    n_folds=5, num_rounds=5, epochs_per_client=50, lam=LAM,
    region_zscore=REGION_ZSCORE, seed=42)
save_results((res3, s3), tag="regionz_3level")

---
## Experiment 7 — Gene subsetting  *(~40 min)*

**CONFIG:** defaults.

Keeps the top-100 severity-correlated genes (selected on training rows only).
Cuts the trunk from 708k to 102k parameters while keeping per-gene weighting —
unlike pooling, which removes it.

Utility holds (MCC 0.191 vs 0.212 with all 691 genes), so ~86% of genes
contribute nothing measurable. This is also the gate for the DP experiment.

In [ ]:
res100, s100 = run_federated_severity(
    data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
    n_folds=5, num_rounds=5, epochs_per_client=50, lam=LAM,
    region_zscore=REGION_ZSCORE, top_k_genes=100, seed=42)
save_results((res100, s100), tag="top100_nodp")

---
## Experiment 8 — Differential privacy  *(~1.5 h)*

**CONFIG:** `DP_SIGMA = 1.5`, then restart and rerun setup.

Per-example gradient clipping plus Gaussian noise, with an RDP accountant.
Note batch-level `clip_grad_norm_` — what the original pipeline does — gives
**no** privacy guarantee regardless of noise.

Result: utility collapses (MCC 0.003) at eps 27.8. Since Experiment 7 showed
the same 102k-parameter model works fine without noise, the failure is cohort
size (~50 patients/site), not capacity.

In [ ]:
assert DP_SIGMA > 0, "set DP_SIGMA = 1.5 in CONFIG, restart, rerun setup"

res_dp, s_dp = run_federated_severity(
    data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
    n_folds=5, num_rounds=3, epochs_per_client=30, lam=LAM,
    region_zscore=REGION_ZSCORE, top_k_genes=100, batch_size=16, seed=42)
save_results((res_dp, s_dp), tag=f"top100_dp_sigma{DP_SIGMA}")

print("\nepsilon:", round(compute_epsilon(
    n_samples=48, batch_size=16, epochs=30, rounds=3, sigma=DP_SIGMA), 2))

---
## Experiment 9 — Pooling ablation  *(~40 min)*

**CONFIG:** `POOL_GENES = True`, `POOL_MODE = "mean"` or `"attention"`,
then restart and rerun setup.

Tests whether shrinking the head to ~1.5k parameters (which would make DP
viable) preserves utility. It does not — both modes collapse with **zero
noise** (MCC 0.05 and 0.01 vs 0.21 flattened). Performance lives in the
707k-parameter layer.

In [ ]:
assert POOL_GENES, "set POOL_GENES = True in CONFIG, restart, rerun setup"

res_p, s_p = run_federated_severity(
    data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
    n_folds=5, num_rounds=5, epochs_per_client=50, lam=LAM,
    region_zscore=REGION_ZSCORE, seed=42)
save_results((res_p, s_p), tag=f"pool_{POOL_MODE}_nodp")

---
## Export results

In [ ]:
import shutil, os
shutil.make_archive("/kaggle/working/results_all", "zip",
                    "/kaggle/working/results")
print("files:")
for f in sorted(os.listdir("/kaggle/working/results")):
    print("  ", f)
print("\n-> download /kaggle/working/results_all.zip")
print("Save Version immediately — /kaggle/working is wiped on restart.")